# Nova Intern 1.5B - Fine-Tuning Pipeline (Unsloth)
This notebook trains the `Nova Intern` persona using the verified MBPP dataset (`dataset_nova_intern_v4.jsonl`).
It uses **Unsloth** for 2x faster training and 70% less memory on Colab free-tier GPUs (T4).

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Good enough for single-function execution
dtype = None # Auto detection
load_in_4bit = True # 4bit quantization to fit on T4 GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct", # Base model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank for LoRA/DoRA
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### Dataset Formatting
Make sure `dataset_nova_intern_v4.jsonl` is uploaded to the Colab files pane!

In [ ]:
from datasets import load_dataset

# Load our strictly formatted JSONL dataset
dataset = load_dataset("json", data_files="dataset_nova_intern_v5.jsonl", split="train")

chat_template = """<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>"""

def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        prompt = messages[0]["content"]
        response = messages[1]["content"]
        text = chat_template.format(prompt=prompt, response=response) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True, remove_columns=dataset.column_names)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 0, # Set to > 0 for quick test, or num_train_epochs = 1 for full run
        num_train_epochs = 1, # Uncomment for a full run!
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start training!
trainer_stats = trainer.train()

### Exporting for Apple Silicon (Mac M3)
Since you are running this locally on a Mac, you need the model in GGUF format so Ollama/llama.cpp can run it at blazing speeds using Metal.

In [ ]:
# Export to Q4_K_M GGUF (Optimized for Mac M3)
model.save_pretrained_gguf("nova_intern_1.5b_gguf", tokenizer, quantization_method = "q4_k_m")

print("✅ GGUF Export Complete! Download the `.gguf` file from the left panel.")